# 09 - Reader: Grading the Test Answers

This notebook gives every test answer of notebook 08 one of the following four labels:

- `correct` : the answer matches the reference answer
- `wrong` : it gives a different answer
- `refusal` : the model said the answer is not on the pages
- `cut off` : the answer reached 1024 tokens before a final answer, it counts as wrong but is kept apart so it can be reported

Code gives the labels it can. The remaining answers gets hand-checked. 

In [1]:
import json

import numpy as np
import pandas as pd

from financebench_rag.data import REPO, load_questions, load_split
from financebench_rag.grading import code_label

SETTINGS = ["gold_text", "gold_image", "retrieved_text", "retrieved_image", "closed_book"]

questions = {q["financebench_id"]: q for q in load_questions()}
test_ids = load_split()["test"]
answers = {s: json.load(open(REPO / "results" / f"test_answers_{s}.json")) for s in SETTINGS}
print({s: len(a) for s, a in answers.items()})

{'gold_text': 120, 'gold_image': 120, 'retrieved_text': 120, 'retrieved_image': 120, 'closed_book': 120}


## 1. Labels given by code

Our code labels an answer in three cases:

- There is no final answer line, so the answer was cut off
- The final answer is exactly `not found`, so it is a refusal
- The question is a metrics question and the final answer is within 3% of the reference answer, so it is correct

Every other answer is left for hand-grading. This includes metrics answers that fail the 3% check, since some of them give the right value in other units, for example 79.8% for a reference of 0.8.

In [2]:
rows = []
for s in SETTINGS:
    for fid in test_ids:
        answer = answers[s][fid]
        rows.append({"setting": s, "financebench_id": fid, "label": code_label(answer, questions[fid]), "final": answer["final"]})
labels = pd.DataFrame(rows)

labels.groupby("setting")["label"].value_counts(dropna=False).unstack(fill_value=0).reindex(SETTINGS)

label,correct,cut off,refusal,NaN
setting,,,,
gold_text,35,7,11,67
gold_image,35,11,8,66
retrieved_text,26,14,25,55
retrieved_image,25,10,24,61
closed_book,1,22,10,87


The code labels 264 of the 600 answers, the other 336 go to the grading sheet.

## 2. Grading sheet

The answers without a label are written to `results/grading_sheet.csv`, one row each, with the question, the reference answer and the model's final answer. The rows are shuffled and do not show the setting, so the grader cannot tell which setting an answer comes from. The link from a row to its setting is kept in `results/grading_key.csv`.

In [3]:
to_grade = labels[labels["label"].isna()].sample(frac=1, random_state=0).reset_index(drop=True)
to_grade["row"] = np.arange(1, len(to_grade) + 1)

sheet = pd.DataFrame({
    "row": to_grade["row"],
    "question": [questions[fid]["question"] for fid in to_grade["financebench_id"]],
    "reference": [questions[fid]["answer"] for fid in to_grade["financebench_id"]],
    "answer": to_grade["final"],
})
sheet.to_csv(REPO / "results" / "grading_sheet.csv", index=False, encoding="utf-8-sig")
to_grade[["row", "setting", "financebench_id"]].to_csv(REPO / "results" / "grading_key.csv", index=False)
print(f"{len(sheet)} answers to grade")

336 answers to grade


## 3. Grading rules

Every row is labelled by these rules.

- `correct` means the answer says the same as the reference. Different wording, rounding and units are fine, as long as the value is the same. Extra explanation is fine unless it contradicts the reference.
- `wrong` means a different value, the opposite yes or no, or a missing part that the question asks for.
- `refusal` means the answer says the information is not available.
- When the final answer is only a number but the question asks "which item" or "yes or no", the conclusion of the full response is graded.
- For "yes or no" questions, the yes or no decides, as long as the response concludes it.

We use an LLM to label our answers for every row, in the column `suggested`. We then hand check LLM's answers and we correct any wrong labels and write the final labeling into the `label` column. The graded sheet is saved into `results/grading_labels.csv`.

## 4. Final labels

The hand checked labels are combined with the code labels, so that every test answer has one label. They are saved to `results/test_labels.csv` for notebook 10. The table counts the labels per setting.

In [7]:
checked = pd.read_csv(REPO / "results" / "grading_labels.csv", sep=None, engine="python", encoding="utf-8-sig")
checked["label"] = checked["label"].str.strip().str.lower()

key = pd.read_csv(REPO / "results" / "grading_key.csv").merge(checked[["row", "label"]], on="row")
final = labels.drop(columns="final").merge(key[["setting", "financebench_id", "label"]], on=["setting", "financebench_id"], how="left", suffixes=("", "_checked"))
final["graded_by"] = np.where(final["label"].isna(), "hand", "code")
final["label"] = final["label"].fillna(final["label_checked"])
final = final.drop(columns="label_checked")
final.to_csv(REPO / "results" / "test_labels.csv", index=False)

counts = final.groupby("setting")["label"].value_counts().unstack(fill_value=0).reindex(SETTINGS)
counts["accuracy"] = (counts["correct"] / len(test_ids)).round(3)
counts

label,correct,cut off,refusal,wrong,accuracy
setting,,,,,
gold_text,94,7,11,8,0.783
gold_image,96,11,8,5,0.800
retrieved_text,67,14,25,14,0.558
retrieved_image,68,10,24,18,0.567
closed_book,22,22,10,66,0.183


### Summary of Results

- Moving from evidence pages to retrieved pages lowers accuracy for more than switching between text and image.
- Without pages, the model answers few questions correctly, so the correct answers from the other settings come from the pages and not from memory.
- Next notebook will test the reliability of differences in results, with confidence interval and question-by-question comparisons.
